# File: **energy_balance.csv**

## Description

This notebook explores the `energy_balance.csv` file produced by PyPSA in each run's results folder. Every row of the CSV corresponds to a `(component, carrier, bus_carrier)` triple with its energy/CO2 value in MWh or tCO2 (sign indicates input/output on that bus).

**What the notebook does, step by step:**

1. **Load** the CSV into `df` and add an `order` column with the original row position. This column is propagated through every transformation so the final bar order in the plot matches the order of the source CSV.
2. **Assign a group** (`electricity`, `H2`, `gas`, `heat`, `co2`, …) to each row based on its `bus_carrier`, using `dic_group`.
3. **Process each group in `list_groups_plot`** with `process_group`:
   - **Link losses merge:** if a Link has several rows with the same `carrier` within a group and the values have **mixed signs** (one input and one output on different buses of the same group), they are merged into a single `'<carrier> losses'` row with `value` = sum. If all values share the same sign (multi-input or multi-output, e.g. DAC) the rows are kept separate.
   - **Charger/discharger merge:** combines rows that differ only in `'charger'` vs `'discharger'` (batteries, water tanks, …), sums their values and strips the word `charger`/`discharger` from the `carrier`.
4. **Global threshold filter:** a `(component, carrier)` is kept if it passes the threshold in **at least one** group where it appears. Anything that fails the threshold in every group is aggregated into a synthetic `'others'` row per group (shown as a final bar).
5. **Plot:** one figure with one row of bars per group, shared x-axis. Bar order follows the original `order`. Colors are read from `config/plotting.default.yaml` (`tech_colors`). Each group is rescaled with `dic_unit_change_plot` and its y-axis labelled via `dic_units_plot`. Groups sharing the same unit **share the vertical range** to make visual comparison easier.

   Bar labels follow a **per-group conflict** rule. A carrier is labelled simply `carrier` when, within every group it appears in, it only has one component — even if that component differs across groups (e.g. `'land transport oil'` is a `Load` in the `oil` subplot and a `Link` in the `co2_atm` subplot; one bar row aligns the demand with its emissions). A carrier is labelled `{component} {carrier}` only when within some single group it appears with multiple components (e.g. `'gas'` with both a `Generator` and a `Load` in the `gas` group); the components are then kept as separate bars in every subplot so the individual contributions stay visible.

In [ ]:
######################################## Parameters

### Run
name = ''
prefix = ''

### Network
clusters = ''
opts = ''
sector_opts = ''
horizon = ''

In [ ]:
##### Import packages
import os
import sys
import warnings
import pandas as pd
import matplotlib.pyplot as plt


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
warnings.filterwarnings('ignore', category=UserWarning)

Load file and show its content.

In [ ]:
df = xp.load_file_csv(
    params,
    filename=f'energy_balance_s_{clusters}_{opts}_{sector_opts}_{horizon}.csv',
    location='results',
    prefix=prefix,
    name=name,
    folder='csvs/individual/',
    skiprows=1,
    header=None,
    names=['component', 'carrier', 'bus_carrier', 'value'],
)

df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df.dropna(subset=['value']).reset_index(drop=True)

df.head()

## bar summary

Show in a bar plot the balance per item and group.

In [ ]:
#################### Parameters

plot_orientation = 'horizontal'  # 'vertical' or 'horizontal'


### Define groups according to bus_carrier.
dic_group = {
    'electricity': ['AC', 'DC', 'low voltage',
                    'battery', 'EV battery', 'home battery'],
    'H2': ['H2'],
    'gas': ['gas', 'biogas','gas for industry'],
    'solid biomass': ['solid biomass', 'solid biomass for industry'],
    'oil': ['oil', 'land transport oil', 'kerosene for aviation','shipping oil',
            'oil primary', 'naphtha for industry', 'agriculture machinery oil'],
    'coal': ['coal', 'coal for industry'],
    'heat': ['urban central heat', 'urban decentral heat', 'rural heat',
             'urban central water tanks', 'urban decentral water tanks', 'rural water tanks',
             'urban central water pits'],
    'co2_captured': ['co2 stored', 'co2 sequestered', 'process emissions'],
    'co2_atm': ['co2'],
    'NH3': ['NH3'],
    'methanol': ['methanol', 'shipping methanol', 'industry methanol'],
}


### Threshold to ignore items (key: threshold value, value: list of groups it applies to)
dic_thresholds = {
    10e6: ['electricity', 'H2', 'gas', 'solid biomass', 'oil', 'heat', 'coal', 'co2_captured', 'co2_atm', 'NH3', 'methanol'],  # MWh / tons
}


### Groups to plot
list_groups_plot = [
    'electricity',
    'H2',
    'gas',
    'solid biomass',
    'oil',
    'coal',
    'heat',
    'co2_captured',
    'co2_atm',
    'NH3',
    'methanol'
]


### Unit change in plot
dic_unit_change_plot = {
    'electricity': 1e-6,  # MWh -> TWh
    'H2': 1e-6,  # MWh -> TWh
    'gas': 1e-6,  # MWh -> TWh
    'solid biomass': 1e-6,  # MWh -> TWh
    'oil': 1e-6,  # MWh -> TWh
    'coal': 1e-6,  # MWh -> TWh
    'heat': 1e-6,  # MWh -> TWh
    'co2_captured': 1e-6,  # tons -> Mt
    'co2_atm': 1e-6,  # tons -> Mt
    'NH3': 1e-6,  # MWh -> TWh
    'methanol': 1e-6,  # MWh -> TWh
}

dic_units_plot = {
    'electricity': 'TWh',
    'H2': 'TWh',
    'gas': 'TWh',
    'solid biomass': 'TWh',
    'oil': 'TWh',
    'coal': 'TWh',
    'heat': 'TWh',
    'co2_captured': 'Mt',
    'co2_atm': 'Mt',
    'NH3': 'TWh',
    'methanol': 'TWh',
}

In [ ]:
### Assign group to each row based on bus_carrier
df['group'] = df['bus_carrier'].apply(lambda bc: xp.assign_group(bc, dic_group))


### Assign order
# 'order' preserves the original CSV row position. It is carried through every
# subsequent transformation (filter by group, Link-losses merge, charger/discharger
# merge, threshold) so the final plot can lay out bars in source-CSV order rather
# than per-group appearance order. On every merge, the first row's 'order' is kept.
df['order'] = range(len(df))

df.head()

In [ ]:
### Process each group: filter + Link-losses merge + charger/discharger merge,
### then apply the global threshold (a key is kept if it passes the threshold
### in at least one group; otherwise aggregated into a per-group 'others' bar).

# Invert dic_thresholds: {group: threshold}
_group_threshold = {g: thr for thr, groups in dic_thresholds.items() for g in groups}

# Phase 1: per-group transforms (no threshold yet)
pre_dfs = {g: xp.process_group(df, g) for g in list_groups_plot}

# Phase 2: global threshold + aggregation of sub-threshold rows into 'others'
dfs_by_group = xp.apply_global_threshold(pre_dfs, _group_threshold, sentinel_order=len(df))


In [ ]:
### Bar charts: one subplot per group, shared carrier order.
### Layout is controlled by plot_orientation ('vertical' = subplots stacked,
### 'horizontal' = subplots side by side with horizontal bars).

# Load tech colors
plotting_cfg = xp.load_file_yaml(params, filename='plotting.default.yaml', location='config')
tech_colors = plotting_cfg['plotting']['tech_colors']
color_fallback = '#999999'

# A carrier is "conflicting" if WITHIN SOME GROUP it appears with more than one
# component (e.g. 'gas' has Generator + Load rows in the 'gas' group). In that
# case the same x-position would otherwise sum heterogeneous flows into a single
# bar, hiding the individual contributions. Conflicting carriers are therefore
# kept split by component in every subplot (label '{component} {carrier}').
#
# A carrier that has only one component within each group it appears in — even
# if the component differs across groups (e.g. 'land transport oil' appears as
# Load in 'oil' and as Link in 'co2_atm') — is NOT conflicting: each subplot
# already disambiguates by its unit and title, and showing it as a single row
# helps the eye associate the demand in one subplot with the emissions in
# another.
_components_in_group = {}  # carrier -> {group: set of components}
for g in list_groups_plot:
    for _, row in dfs_by_group[g].iterrows():
        _components_in_group.setdefault(row['carrier'], {}).setdefault(g, set()).add(row['component'])

_conflicting = {
    carr
    for carr, by_group in _components_in_group.items()
    if any(len(comps) > 1 for comps in by_group.values())
}

# Build master list of bar keys. For conflicting carriers we use (component, carrier)
# so each component keeps its own bar; for non-conflicting carriers we use (None, carrier)
# so rows of the same carrier with different components in different groups share the
# same bar position. Sort by original CSV order (minimum across groups).
_order_by_key = {}
for g in list_groups_plot:
    for _, row in dfs_by_group[g].iterrows():
        carr = row['carrier']
        key = (row['component'], carr) if carr in _conflicting else (None, carr)
        o = int(row['order'])
        if key not in _order_by_key or o < _order_by_key[key]:
            _order_by_key[key] = o
master_keys = sorted(_order_by_key.keys(), key=lambda k: _order_by_key[k])

labels = [f"{comp} {carr}" if comp is not None else carr for comp, carr in master_keys]
colors = [tech_colors.get(carr, color_fallback) for _, carr in master_keys]

# Per-group values reindexed to master_keys, scaled, missing filled with 0.
# Non-conflicting carriers sum across components within the group; conflicting
# carriers keep components separate.
def _values_for_group(g):
    df_g = dfs_by_group[g]
    by_pair    = df_g.groupby(['component', 'carrier'])['value'].sum()
    by_carrier = df_g.groupby('carrier')['value'].sum()
    scale = dic_unit_change_plot.get(g, 1.0)
    out = []
    for comp, carr in master_keys:
        if comp is None:
            val = by_carrier.get(carr, 0.0)
        else:
            val = by_pair.get((comp, carr), 0.0)
        out.append(float(val) * scale)
    return out

# Precompute scaled values per group and shared value-range per unit
vals_by_group = {g: _values_for_group(g) for g in list_groups_plot}

# For each unit, take the min/max across all groups that share it
unit_range = {}
for g in list_groups_plot:
    unit = dic_units_plot.get(g, 'value')
    vals = vals_by_group[g]
    lo, hi = min(vals + [0.0]), max(vals + [0.0])
    cur_lo, cur_hi = unit_range.get(unit, (lo, hi))
    unit_range[unit] = (min(cur_lo, lo), max(cur_hi, hi))

# Add 5% padding to each unit range
unit_range = {
    u: (lo - 0.05 * (hi - lo) if hi != lo else lo - 1, hi + 0.05 * (hi - lo) if hi != lo else hi + 1)
    for u, (lo, hi) in unit_range.items()
}

n = len(list_groups_plot)

if plot_orientation == 'vertical':
    # n subplots stacked vertically, vertical bars, carriers on x-axis
    fig_width = max(8, 0.5 * len(master_keys))
    fig, axes = plt.subplots(n, 1, figsize=(fig_width, 4 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, g in zip(axes, list_groups_plot):
        vals = vals_by_group[g]
        ax.bar(labels, vals, color=colors, edgecolor='black', linewidth=0.6)
        ax.axhline(0, color='black', linewidth=1)
        ax.set_title(g)
        unit = dic_units_plot.get(g, 'value')
        ax.set_ylabel(unit)
        ax.set_ylim(unit_range[unit])
        ax.grid(axis='y', linestyle='--', alpha=0.35)
        ax.grid(axis='x', linestyle=':', linewidth=0.5, alpha=0.4)
        ax.set_axisbelow(True)

    axes[-1].set_xlabel('carrier')
    axes[-1].tick_params(axis='x', rotation=60)
    plt.setp(axes[-1].get_xticklabels(), ha='right')

elif plot_orientation == 'horizontal':
    # n subplots side by side, horizontal bars, carriers on y-axis (top = first CSV row)
    fig_height = max(8, 0.5 * len(master_keys))
    fig, axes = plt.subplots(1, n, figsize=(4 * n, fig_height), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, g in zip(axes, list_groups_plot):
        vals = vals_by_group[g]
        ax.barh(labels, vals, color=colors, edgecolor='black', linewidth=0.6)
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title(g)
        unit = dic_units_plot.get(g, 'value')
        ax.set_xlabel(unit)
        ax.set_xlim(unit_range[unit])
        ax.grid(axis='x', linestyle='--', alpha=0.35)
        ax.grid(axis='y', linestyle=':', linewidth=0.5, alpha=0.4)
        ax.set_axisbelow(True)

    axes[0].set_ylabel('carrier')
    axes[0].invert_yaxis()  # first carrier at the top (propagates via sharey)

else:
    raise ValueError(f"plot_orientation must be 'vertical' or 'horizontal', got {plot_orientation!r}")

plt.tight_layout()

xp.save_figure(filename='energy_balance.pdf')